In [67]:
%pip install -q cassio datasets langchain openai tiktoken

Note: you may need to restart the kernel to use updated packages.


In [68]:
from langchain.vectorstores.cassandra import Cassandra 
from langchain.indexes.vectorstore import VectorStoreIndexWrapper # wrap vectors in one package
from langchain.llms import Ollama
from langchain.embeddings import OllamaEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

True

In [69]:
# support dataset retrieval  with hugging face 
from datasets import load_dataset
# with cassio the engine powering astra db integration in langchain 
# need to initialize the db connection first
import cassio

In [70]:
from PyPDF2 import PdfReader

In [71]:
pdfreader = PdfReader("project2_midsem_report.pdf")

#### 1) reading the text 

In [72]:
from typing_extensions import Concatenate
# read text from pdf 
raw_text = ""
for i , page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [73]:
raw_text

'B.TECH. PROJECT ON\nDesign and comparative analysis of A\n2-V 10.7-MHz CMOS limiting\namplifier/RSSI\nSubmitted\nBy:RAMAN 2021UEC2565\n2021UEC2564\n2021UEC2627YOGESH GUPTA\nNAMAN NIMBLE\nUnder the Guidance\nof (Dr. Urvashi\nBansal)\nProject- II in partial fulfillment of requirement for\nthe award of B.Tech. in Electronics &\nCommunication Engineering\nDepartment of Electronics & Communication\nEngineering NETAJI SUBHAS UNIVERSITY OF\nTECHNOLOGY NEW DELHI-110078 September-\n2024CERTIFICATE\nDate:23/09/2024Place: New DelhiCertified that Raman (2021UEC2565), Yogesh Gupta (2021UEC2564),Naman\nNimble (2021UEC2627) has carried out their project work presented in this\nproject entitled “A 2-V 10.7-MHz CMOS limiting amplifier/RSSI” for the\naward of Bachelor of Technology, Department of Electronics and\nCommunication, Netaji Subhas University of Technology, New Delhi, under\nmy/our supervision. The project embodies results of original work, and studies\nare carried out by the student himself/

#### 2) initializing the database

In [74]:
cassio.init(
    # Astra DB connection details
    token = os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    database_id = os.getenv("ASTRA_DB_CLIENT_ID"),
)

#### 3) initializing the model and creating the embeddings

In [79]:
llm = Ollama(model="llama2")
# from langchain_groq import ChatGroq

# llm=ChatGroq(model="Gemma2-9b-It",groq_api_key=os.getenv("GROQ_API_KEY"))
embeddings = OllamaEmbeddings(model="llama2")

astra_vector_store = Cassandra(
    embedding=embeddings,
    table_name="qa_mini_demo",
    session=None,
    keyspace=None,
)

#### 4) splitting the text into chunks and embedding them

In [80]:
from langchain.text_splitter import CharacterTextSplitter

text_splitters = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)

texts = text_splitters.split_text(raw_text) # raw text is the text from pdf
texts

['B.TECH. PROJECT ON\nDesign and comparative analysis of A\n2-V 10.7-MHz CMOS limiting\namplifier/RSSI\nSubmitted\nBy:RAMAN 2021UEC2565\n2021UEC2564\n2021UEC2627YOGESH GUPTA\nNAMAN NIMBLE\nUnder the Guidance\nof (Dr. Urvashi\nBansal)\nProject- II in partial fulfillment of requirement for\nthe award of B.Tech. in Electronics &\nCommunication Engineering\nDepartment of Electronics & Communication\nEngineering NETAJI SUBHAS UNIVERSITY OF\nTECHNOLOGY NEW DELHI-110078 September-\n2024CERTIFICATE\nDate:23/09/2024Place: New DelhiCertified that Raman (2021UEC2565), Yogesh Gupta (2021UEC2564),Naman\nNimble (2021UEC2627) has carried out their project work presented in this\nproject entitled “A 2-V 10.7-MHz CMOS limiting amplifier/RSSI” for the\naward of Bachelor of Technology, Department of Electronics and\nCommunication, Netaji Subhas University of Technology, New Delhi, under\nmy/our supervision. The project embodies results of original work, and studies\nare carried out by the student himself

In [81]:
astra_vector_store.add_texts(texts) #  add the text to the vector store
# print("inserted headlines into the vector store", len(texts[:50]))
astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store) # wrap the vector store in a index wrapper

### running the q and a cycle

In [82]:
first_question = True 
while True:
    if first_question:
        question = input("Ask a question: ").strip()
    else:
        question = input("Ask another question: ").strip()
    
    if question.lower() == "exit":
        break
    
    if question == "":
        continue
    
    first_question = False
    print("QUESTION: ", question)
    answer = astra_vector_index.query(question, llm=llm).strip() # query the vector store with the question
    print("ANSWER: ", answer) # print the answer
    
    # print("first document by relevance: ") # print the first document by relevance  
    # for doc , score in astra_vector_index.similarity_search_with_score(question, k=4):
    #     print(doc.page_content)

QUESTION:  what is this project about ?
ANSWER:  This project appears to be focused on the design and simulation of a low power CMOS limiting amplifier and RSSI circuit for high-speed communication applications. The goal is to achieve a high gain, low noise, and low power consumption while operating at 10.7 MHz IF frequency.

The project involves the use of MATLAB/Simulink software for simulation and analysis of the proposed amplifier and RSSI circuit. The key performance parameters such as gain, bandwidth, offset voltage, and power dissipation are analyzed and compared with existing designs to highlight the advantages of the proposed approach.

The project also involves the use of PSpice software for circuit simulation and analysis. The proposed amplifier and RSSI circuit are simulated and their performance is evaluated under various process corners.

Overall, this project seems to be focused on developing a high-performance CMOS limiting amplifier and RSSI circuit for high-speed comm

KeyboardInterrupt: 